# Preprocessing in Python

In [1]:
# Imports
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import networkx as nx  
import sys
sys.path.insert(0 , './../MAIN/')
import preprocess_functions

# ----------------
# Configuration
# ----------------
datadir = Path("/work/gr-fe/bryan/data/YEAST/")  # adjust if needed
cohort = ""
omics = ["EXPR"]

input_dir = datadir / cohort / "02_processed"
output_dir = input_dir  # save alongside inputs
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Input dir:  {input_dir}")
print(f"Output dir: {output_dir}")

# ----------------
# Processing loop
# ----------------
for omic in omics:
    print(f"\n=== Processing {omic} ===")

    expr_path = input_dir / f"{omic}.pkl"
    meta_path = input_dir / "phenotype.processed.pkl"

    # Load expression
    if not expr_path.exists():
        print(f" ! Skipping {omic}: missing {expr_path}")
        continue

    with open(expr_path, "rb") as f:
        obj = pickle.load(f)

    if isinstance(obj, dict) and "expr" in obj:
        expr = obj["expr"]
    else:
        expr = pd.DataFrame(obj)

    # Ensure float32 dtype
    expr = expr.astype(np.float32)

    # Load metadata and align to expr index
    if not meta_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {meta_path}")

    with open(meta_path, "rb") as f:
        meta = pickle.load(f)

    if "ID" not in meta.columns:
        raise KeyError("Expected 'ID' column in phenotype.processed.pkl")

    # Align rows of metadata to expression index
    try:
        meta = meta.set_index("ID").loc[expr.index]
    except KeyError as e:
        missing = set(expr.index) - set(meta["ID"])
        raise KeyError(
            f"Some expression IDs are missing from metadata. Missing count: {len(missing)} "
            f"(e.g., {list(missing)[:5]})"
        ) from e

    # Determine per-omic behavior
    is_transcriptomics = omic in ["mRNA", "miRNA"]
    filter_genes = (omic == "mRNA")
    do_log_transform = (omic in [])

    # Process
    if is_transcriptomics:
        expr, meta = preprocess_functions.data_preprocess(expr, meta, filter_gene_expr=filter_genes)

        vsd = preprocess_functions.DESEQ(expr)
        expr = pd.DataFrame(data=vsd, index=expr.index, columns=expr.columns)
        print(" - Applied DESeq VST")

    elif do_log_transform:
        expr, meta = preprocess_functions.data_preprocess(
            expr, meta, filter_gene_expr=False, log_transform=True
        )
        print(" - Applied log-transform preprocessing")

    else:
        expr, meta = preprocess_functions.data_preprocess(expr, meta)
        print(" - Applied default preprocessing")

    # Save
    out_path = output_dir / f"{omic}.processed.pkl"
    with open(out_path, "wb") as f:
        pickle.dump({"expr": expr, "meta": meta}, f)
    print(f" - Saved: {out_path}")

# Optional: list saved outputs
print("\nProcessed files:")
for p in sorted(output_dir.glob("*.processed.pkl")):
    print(" -", p.name)


Input dir:  /work/gr-fe/bryan/data/YEAST/02_processed
Output dir: /work/gr-fe/bryan/data/YEAST/02_processed

=== Processing EXPR ===
Keeping 2409 Samples
Removed 8 Samples
 - Applied default preprocessing
 - Saved: /work/gr-fe/bryan/data/YEAST/02_processed/EXPR.processed.pkl

Processed files:
 - EXPR.processed.pkl
 - phenotype.processed.pkl
